# US ESG / SRI Fund Name Screener

Sibling of `Impact_Fund_Screener_v8.ipynb`, but three deliberate differences:

1. **Scope is inverted.** The v8 impact screener excludes generic ESG/SRI language
   (it is trying to isolate *intentional impact* funds). This screener is the broad
   ESG/SRI net — `ESG` and `SRI` are the first two matching terms, not stop-tokens.
2. **English-only.** All multilingual patterns and non-English objective columns are
   dropped; this runs against the US equity Morningstar dataset.
3. **No blocking tokens.** `KNOWN_NEGATIVE_ABBREVS` is empty by design (see Cell 3).
   Nothing suppresses a match. Recall-first; false positives are a downstream-review
   problem, false negatives are never seen again.

**Term source:** every row of `Regex_Terms_for_US_ESG-SRI_funds.xlsx` (53 rows).
Cleaning applied before compiling — all changes are annotated inline in Cell 2:
leading/trailing spaces trimmed; `positive change` (listed twice) de-duplicated;
`transformative` folded into the `transform\w*` pattern with `transformation`;
`bBetter world` (typo) corrected to `better world`; slash-variants
(`net zero / net-zero`, `Paris-aligned / Paris aligned`, `fossil free / fossil-free`)
each compiled as a single pattern.

**Matching is word-boundary-aware**, not substring. This is *not* blocking — it is
correct tokenisation. It is the only thing standing between this screener and a flood
of `Evergreen`, `Global Economy`, `Diversified`, `Signature`, and `Value` funds.
Validated on a false-friend regression set (see notes at the bottom of Cell 2).

## CELL 1 — Configuration

In [ ]:
import os, re, json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# Same File_Directory.txt indirection as the rest of the project.
# IMPORTANT: point "Input" at the US equity Morningstar file, NOT an EU file.
# (Sanity-check the output fund families are US names — Calvert, Parnassus,
#  Nuveen, Domini, Pax, TIAA — not European ones.)
config = {}
with open("File_Directory.txt") as f:
    for line in f:
        if ":" in line:
            key, val = line.split(":", 1)
            config[key.strip()] = val.strip()

INPUT_FILE  = Path(config["Input"])
OUTPUT_DIR  = Path(config["Output"])

NAME_COL    = "Name"
ID_COL      = "FundId"

# English-only. Cell 6 keeps only the columns that actually exist in the file,
# so listing a superset here is safe. These columns are carried into the output
# for context only — matching runs on the fund NAME, not on these.
OBJECTIVE_COLUMNS = [
    "Prospectus Objective",
    "KIID Objective/Investment Policy",
    "PRIIPS KID Objective",
    "Strategy Description",
    "Investment Strategy - English",
]

print("Configuration loaded.")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")


## CELL 2 — Keyword Patterns (all 53 xlsx terms, English-only)

In [ ]:
# Each entry: (concept, label, regex_pattern, needs_review)
#
# needs_review=True  -> match flagged for human inspection (ambiguous / FP risk).
# re.IGNORECASE applied at compile time to every pattern.
#
# Every row of Regex_Terms_for_US_ESG-SRI_funds.xlsx is represented below.
# Wildcards from the sheet (term*) become \w*. Bare single words become \bword\b
# so they match a WHOLE word, not a substring. The trailing "# xlsx rN" comment
# is the source row. Deviations from a literal port are called out with "# NOTE".
FLAGS = re.IGNORECASE

PATTERNS = [
    # ===== CORE LABELS =====
    ("ESG",                 "esg",                r"\bESG\b",                       False),  # xlsx r1
    ("SRI",                 "sri",                r"\bSRI\b",                       False),  # xlsx r2  (Dirk 2026-06-23: SRI in scope)

    # ===== BROAD ESG VOCABULARY =====
    ("sustainable",         "sustainab",          r"\bsustainab\w*",                False),  # xlsx r3   sustainable/sustainability/sustainably
    ("responsible",         "responsib",          r"\bresponsib\w*",                False),  # xlsx r4   responsible/responsibly/responsibility
    ("social",              "social",             r"\bsocial\b",                    False),  # xlsx r5   NOTE: bare word; "socially responsible" also caught by responsib
    ("ethical",             "ethic",              r"\bethic\w*",                    False),  # xlsx r6   ethic/ethics/ethical/ethically
    ("justice",             "justice",            r"\bjustice\b",                   False),  # xlsx r7
    ("equality",            "equality",           r"\bequality\b",                  False),  # xlsx r8   NOTE: distinct from 'equity'; does not match Eq/Equity
    ("impact",              "impact",             r"\bimpact",                      False),  # xlsx r9   NOTE: prefix (no closing \b) -> impact/impactful/Impact360/impacto
    ("green",               "green",              r"\bgreen\b",                     False),  # xlsx r10  NOTE: \b rejects Evergreen/Greenwich/greenfield
    ("climate",             "climate",            r"\bclimate\b",                   False),  # xlsx r11
    ("carbon",              "carbon",             r"\bcarbon\b",                    False),  # xlsx r12  NOTE: \b rejects hydrocarbon/carbonate/Carbonite
    ("decarbonisation",     "decarbon",           r"\bdecarbon\w*",                 False),  # xlsx r13  decarbon(ise|ize|isation|ization)
    ("net zero",            "net_zero",           r"\bnet[\s\-]?zero\b",            False),  # xlsx r14  "net zero / net-zero / netzero"
    ("Paris-aligned",       "paris_aligned",      r"\bparis[\s\-]aligned\b",        False),  # xlsx r15  "Paris-aligned / Paris aligned"
    ("environmental",       "environment",        r"\benvironment\w*",              False),  # xlsx r16  environment/environmental/environmentally
    ("eco",                 "eco",                r"\beco(?!nom)\w*",               True),   # xlsx r17  NOTE: excludes econom- (economy/economic/economies) — see bottom of cell. Still catches ecological/ecosystem/eco-/Ecofin
    ("governance",          "governance",         r"\bgovernance\b",                False),  # xlsx r18
    ("SDG",                 "sdg",                r"\bSDGs?\b",                     False),  # xlsx r19
    ("renewable",           "renewable",          r"\brenewab\w*",                  False),  # xlsx r20  renewable/renewables
    ("biodiversity",        "biodivers",          r"\bbiodivers\w*",                False),  # xlsx r21  biodiversity/biodiverse
    ("fossil-free",         "fossil_free",        r"\bfossil[\s\-]?free\b",         False),  # xlsx r22  "fossil free / fossil-free"
    ("values",              "values",             r"\bvalues\b",                    False),  # xlsx r23  NOTE: plural only -> rejects ubiquitous 'Value' funds; catches values-aligned/values-based
    ("clean",               "clean",              r"\bclean\b",                     False),  # xlsx r24  NOTE: \b rejects cleaning/cleanse; catches "clean energy/tech/power". Solid 'cleantech' NOT caught
    ("solar",               "solar",              r"\bsolar\b",                     False),  # xlsx r25
    ("wind",                "wind",               r"\bwind\b",                      False),  # xlsx r26  NOTE: \b rejects window/winding/windfall/rewind
    ("hydrogen",            "hydrogen",           r"\bhydrogen\b",                  False),  # xlsx r27
    ("battery",             "battery",            r"\bbatter(?:y|ies)\b",           True),   # xlsx r28  NOTE: matches place-name "Battery Park" too (intrinsic FP, review downstream)
    ("new energy",          "new_energy",         r"\bnew[\s\-]energy\b",           False),  # xlsx r29
    ("alternative energy",  "alternative_energy", r"\balternative[\s\-]energy\b",   False),  # xlsx r30
    ("energy transition",   "energy_transition",  r"\benergy[\s\-]transition\b",    False),  # xlsx r31  NOTE: also caught by bare 'transition' (r50); kept for its own label
    ("positive change",     "positive_change",    r"\bpositive[\s\-]change\b",      False),  # xlsx r32 (== r43, de-duplicated)
    ("positive future",     "positive_future",    r"\bpositive[\s\-]future\b",      False),  # xlsx r33 (leading space trimmed)
    ("norms-based",         "norms_based",        r"\bnorms?[\s\-]based\b",         False),  # xlsx r34  norms-based / norm-based
    ("UN Global Compact",   "un_global_compact",  r"\bUN[\s\-]global[\s\-]compact\b",False),  # xlsx r35 (leading space trimmed)
    ("UNGC",                "ungc",               r"\bUNGC\b",                      False),  # xlsx r36 (leading space trimmed)
    ("nature",              "nature",             r"\bnature\b",                    False),  # xlsx r37  NOTE: \b rejects Signature/nomenclature; excludes 'natural' (Natural Resources funds)
    ("greenhouse gas",      "greenhouse_gas",     r"\bgreenhouse[\s\-]gas\w*",      False),  # xlsx r38  greenhouse gas/gases
    ("GHG",                 "ghg",                r"\bGHG\b",                       False),  # xlsx r39
    ("diversity",           "diversity",          r"\bdiversity\b",                 False),  # xlsx r40  NOTE: exact -> rejects diversified/diverse; biodiversity handled by r21
    ("empowerment",         "empowerment",        r"\bempower\w*",                  False),  # xlsx r41  empower/empowering/empowerment
    ("women",               "women",              r"\bwomen\b",                     False),  # xlsx r42  women / women's
    # xlsx r43 "positive change" -> duplicate of r32, folded above
    ("better world",        "better_world",       r"\bbetter[\s\-]world\b",         False),  # xlsx r44  NOTE: sheet literally reads "bBetter world" (typo) — corrected
    ("better future",       "better_future",      r"\bbetter[\s\-]future\b",        False),  # xlsx r45
    ("future generations",  "future_generations", r"\bfuture[\s\-]generations?\b",  False),  # xlsx r46
    ("future for generations","future_for_gens",  r"\bfuture[\s\-]for[\s\-]generations?\b", False),  # xlsx r47
    ("transformation",      "transform",          r"\btransform\w*",                False),  # xlsx r48 + r49  NOTE: covers transformation AND transformative (and transforming)
    ("transition",          "transition",         r"\btransition\b",                False),  # xlsx r50  NOTE: low precision — matches "Transition Materials" etc.
    ("engagement",          "engagement",         r"\bengagement\b",                False),  # xlsx r51  (abbreviations Enga/Engmnt caught via Cell 3 expansion)
    ("active ownership",    "active_ownership",    r"\bactive[\s\-]ownership\b",     False),  # xlsx r52
    ("stewardship",         "steward",            r"\bsteward\w*",                  False),  # xlsx r53  steward/stewards/stewardship/stewarding
]

COMPILED_PATTERNS = [
    (concept, label, re.compile(pat, FLAGS), new)
    for concept, label, pat, new in PATTERNS
]

n_concepts = len({c for c, *_ in PATTERNS})
n_review   = sum(1 for *_, new in PATTERNS if new)
print(f"Loaded {len(COMPILED_PATTERNS)} patterns across {n_concepts} concepts "
      f"({n_review} flagged needs_review).")
print("All 53 xlsx rows accounted for: r43 == r32 (de-duplicated), "
      "r49 folded into r48 (transform\\w*).")

# --------------------------------------------------------------------------
# FALSE-FRIEND REGRESSION (why the \b handling above is not optional).
# Run this block to confirm boundary handling before a full dataset run.
# Expected: 0 recall misses, and the ONLY should-not-match hit is the
# intrinsic "Battery Park" case (term 'battery' is genuinely present).
# --------------------------------------------------------------------------
if __name__ == "__main__":
    _hit = lambda s: [c for c, _l, rx, _n in COMPILED_PATTERNS if rx.search(s)]
    _should = ["Calvert Green Bond","Invesco Solar ETF","Global Wind Energy",
               "iShares Clean Energy","Ecofin Global Renewables","Domini Impact Eq",
               "Nature Conservancy","Values-Aligned Eq","Biodiversity Leaders",
               "Decarbonization ETF","VanEck Low Carbon"]
    _shouldnt = ["Evergreen Income","Greenwich Growth","Vanguard Value","Diversified Intl",
                 "Global Economy Opps","Economic Recovery","Signature Select",
                 "Natural Resources","Natural Gas","Window Rock","Windfall Trust",
                 "Hydrocarbon ETF","Carbonite Fund","Cleanse Wellness","Cleaning REIT"]
    miss = [s for s in _should if not _hit(s)]
    fp   = [(s, _hit(s)) for s in _shouldnt if _hit(s)]
    print(f"\nRegression: {len(_should)-len(miss)}/{len(_should)} should-match OK; "
          f"{len(fp)} should-not-match hits {fp}")


## CELL 3 — Abbreviation Expansion Map (NO blocking tokens)

In [ ]:
# Token-level expansion: an ABBREVIATED token -> the full word, so the
# Cell 2 patterns can fire on the reconstructed name (Pass 2 in Cell 6).
# Lookup is case-insensitive; the value is the canonical output spelling.
#
# This map is intentionally ESG-CONCEPT-FOCUSED. Only tokens that, once
# expanded, can actually trigger a Cell-2 pattern are worth having here.
# (A generic "Glb"->"Global" expansion is cosmetic — Global is not an ESG
#  term — so only a small readability set of those is kept, clearly grouped.)
ABBREV_EXPANSION = {
    # ── ESG concepts (these CAN trigger a match after expansion) ──
    "Sust": "Sustainable", "Sus": "Sustainable", "Sstn": "Sustainable", "Sustn": "Sustainable",
    "Env": "Environmental", "Envir": "Environmental", "Enviro": "Environmental",
    "Soc": "Social", "Scl": "Social",
    "Gov": "Governance", "Govn": "Governance",
    "Clmt": "Climate", "Clim": "Climate", "Clm": "Climate",
    "Imp": "Impact", "Impct": "Impact", "Impctf": "Impact",
    "Renew": "Renewable", "Rnwbl": "Renewable", "Renwbl": "Renewable",
    "Trans": "Transition", "Trnstn": "Transition", "Trnsn": "Transition",
    "Transf": "Transformation",
    "Enga": "Engagement", "Engmnt": "Engagement", "Enggmnt": "Engagement", "Engmt": "Engagement",
    "Grn": "Green",
    "Pos": "Positive", "Pstv": "Positive", "Postv": "Positive",
    "Btr": "Better", "Bttr": "Better",
    "Chng": "Change", "Chg": "Change",
    "Gens": "Generations", "Gen": "Generation",
    "Divrsty": "Diversity", "Dvrsty": "Diversity",

    # ── Generic readability only (never create an ESG match) ──
    "Glb": "Global", "Glbl": "Global", "Fd": "Fund", "Fds": "Funds",
    "Mkt": "Market", "Mkts": "Markets", "Intl": "International", "Emg": "Emerging",
    "Eq": "Equity", "Eqs": "Equities",
}
ABBREV_EXPANSION_CI = {k.lower(): v for k, v in ABBREV_EXPANSION.items()}

# ============================================================================
# NO BLOCKING TOKENS  (per request).
# This dict is deliberately EMPTY. In the v8 impact screener it held
# ambiguous tokens (Dev/Eng/Act/Transp/Akt/CA...) that were prevented from
# expanding into a match. Here nothing is suppressed.
#
# Verified safe: none of those ex-blocked tokens appears in ABBREV_EXPANSION
# above in a form that maps to an ESG concept, so emptying this list creates
# zero new expansion-path false positives. (Regression-checked.)
# ============================================================================
KNOWN_NEGATIVE_ABBREVS = {}
KNOWN_NEGATIVE_ABBREVS_CI = {k.lower(): v for k, v in KNOWN_NEGATIVE_ABBREVS.items()}

def expand_name(fund_name: str):
    """Best-effort token expansion.
    Returns (expanded_name, fully_expanded, flagged_notes).
    With no blocking tokens, flagged_notes is always empty; fully_expanded is
    False only when an UNKNOWN all-caps acronym is left un-expanded (metadata
    for the reviewer — it does NOT block the match)."""
    tokens = re.split(r'([\s\-–&/()+]+)', fund_name)
    expanded_tokens, fully_expanded, flagged_notes = [], True, []
    for token in tokens:
        stripped = token.strip()
        stripped_ci = stripped.lower()
        if stripped_ci in ABBREV_EXPANSION_CI:
            expanded_tokens.append(ABBREV_EXPANSION_CI[stripped_ci])
        elif stripped_ci in KNOWN_NEGATIVE_ABBREVS_CI:      # never fires (empty)
            expanded_tokens.append(token); fully_expanded = False
            flagged_notes.append(f"{stripped}: {KNOWN_NEGATIVE_ABBREVS_CI[stripped_ci]}")
        elif re.match(r'^[A-Z]{2,5}$', stripped) and stripped not in {
            "USD","EUR","GBP","CHF","JPY","CAD","AUD",   # currencies
            "ETF","UCITS","CEF","BDC",                    # US fund types
            "ESG","SRI","SDG","GHG","UNGC",               # already-known ESG acronyms (Cell 2)
            "ACC","INC","CAP","DIS","INST","ADV",         # share class
            "US","USA","EM","EU","UK","NA",               # geography
            "AI","IT","IP","REIT",                        # tech / structure
        }:
            expanded_tokens.append(token); fully_expanded = False  # metadata only
        else:
            expanded_tokens.append(token)
    return "".join(expanded_tokens), fully_expanded, flagged_notes


## CELL 4 — Token Scan (exploratory)

In [ ]:
# US-oriented reference set of tokens confirmed to carry no ESG meaning.
# Used ONLY by the scan below to surface UNKNOWN acronyms in the dataset so
# the ABBREV_EXPANSION map can be extended. NOT used in matching -> not blocking.
SAFE_TOKENS = {
    # Currencies
    "USD","EUR","GBP","CHF","JPY","CAD","AUD","HKD",
    # US fund structures
    "ETF","UCITS","CEF","BDC","REIT","MLP","LP","LLC","INC",
    # Share class / series
    "A","B","C","D","F","I","K","N","R","T","Y","Z",
    "ACC","CAP","DIS","ADV","INST","INV","RET","SVC","ADM","IX",
    # Geography
    "US","USA","EM","EU","UK","EAFE","NA","INTL",
    # Style / strategy descriptors (non-ESG)
    "EQ","RE","SMID","LC","MC","SC","GR","VAL","AGG","IG","HY","TIPS",
    # Common large US manager / brand codes
    "GS","JPM","MS","BR","MFS","TRP","AB","DFA","SPDR","TIAA","CREF","PGIM",
}

# SRI confirmed in scope by Dirk (2026-06-23); ESG/SRI/SDG/GHG/UNGC are matching
# terms in Cell 2, so they are intentionally NOT in SAFE_TOKENS.


In [ ]:
# Run once on the full dataset to surface unknown abbreviations
# before finalising the ABBREV_EXPANSION map.
print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  {len(df)} funds, {len(df.columns)} columns")

all_tokens = []
for name in df[NAME_COL].dropna():
    tokens = re.split(r'[\s\-–&/()+]+', str(name))
    all_tokens.extend(t for t in tokens if t)

token_counts = Counter(all_tokens)
unknown_abbrevs = [
    (tok, count) for tok, count in token_counts.most_common(500)
    if re.match(r'^[A-Z]{2,5}$', tok) and tok not in SAFE_TOKENS and count >= 2
]

print(f"\n{'Token':<12} {'Count':>6}   (known expansion)")
print("-" * 45)
for tok, count in unknown_abbrevs[:40]:
    print(f"{tok:<12} {count:>6}   {ABBREV_EXPANSION.get(tok, '?')}")


## CELL 5 — Matching Function

In [ ]:
def match_fund(fund_name: str) -> list:
    """One record per pattern matched on a single fund name."""
    matches = []
    for concept, label, compiled_re, new in COMPILED_PATTERNS:
        m = compiled_re.search(fund_name)
        if m:
            matches.append({
                "matched_concept":       concept,
                "matched_pattern_label": label,
                "matched_text":          m.group(0),
                "needs_review":          new,
            })
    return matches


## CELL 6 — Run Matching (raw name + abbreviation-expanded name)

In [ ]:
print("Running pattern matching...")

available_obj_cols = [c for c in OBJECTIVE_COLUMNS if c in df.columns]

results = []
n_expansion_only_hits = 0

for _, row in df.iterrows():
    fund_name = str(row.get(NAME_COL, ""))
    fund_id   = row.get(ID_COL, "")

    expanded_name, fully_expanded, flagged_notes = expand_name(fund_name)

    # Pass 1: raw name.
    raw_matches = match_fund(fund_name)
    raw_labels  = {m["matched_pattern_label"] for m in raw_matches}
    combined = [{**m, "Matched_Via": "raw"} for m in raw_matches]

    # Pass 2: expanded name, for every fund. Any label found only via the
    # expanded name is added and force-flagged needs_review=True, since it
    # rests on expand_name()'s best-effort guesses rather than the literal name.
    if expanded_name != fund_name:
        for m in match_fund(expanded_name):
            if m["matched_pattern_label"] not in raw_labels:
                combined.append({**m, "needs_review": True, "Matched_Via": "expansion"})
                n_expansion_only_hits += 1

    if not combined:
        continue

    obj_texts = {col: row.get(col, "") for col in available_obj_cols}
    for match in combined:
        results.append({
            ID_COL:               fund_id,
            NAME_COL:             fund_name,
            "Name_Expanded":      expanded_name,
            "Expansion_Complete": fully_expanded,
            "Flagged_Tokens":     "; ".join(flagged_notes),   # always empty (no blocking)
            **match,
            **obj_texts,
        })

results_df = pd.DataFrame(results)
print(f"  {results_df[ID_COL].nunique()} funds matched across {len(results_df)} pattern hits")
print(f"  of which {n_expansion_only_hits} hits found ONLY via abbreviation expansion "
      f"(all flagged needs_review=True; see Matched_Via column)")


## CELL 7 — Summary Statistics

In [ ]:
if len(results_df) == 0:
    print("No matches found.")
else:
    funds_df = results_df.drop_duplicates(subset=ID_COL)
    print(f"\n{'='*55}")
    print(f"  US ESG/SRI CANDIDATES: {len(funds_df)} of {len(df)} funds "
          f"({len(funds_df)/len(df)*100:.1f}%)")
    print(f"{'='*55}")

    print("\nHits by concept:")
    for label, count in results_df["matched_pattern_label"].value_counts().items():
        n_rev = results_df[results_df["matched_pattern_label"] == label]["needs_review"].sum()
        flag = "  * needs review" if n_rev > 0 else ""
        print(f"  {label:<22} {count:>4}{flag}")

    review_count = int(results_df["needs_review"].sum())
    print(f"\nRows flagged for human review: {review_count} "
          f"({review_count/len(results_df)*100:.1f}% of all hits)")

    exp_hits  = (results_df["Matched_Via"] == "expansion").sum()
    exp_funds = results_df[results_df["Matched_Via"] == "expansion"][ID_COL].nunique()
    print(f"Found ONLY via abbreviation expansion: {exp_hits} hits across {exp_funds} funds")


## CELL 8 — Output to Excel

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile = OUTPUT_DIR / f"US_ESG_SRI_Candidates_{timestamp}.xlsx"

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:
    # Sheet 1 — one row per pattern hit
    results_df.to_excel(writer, sheet_name="Matches", index=False)

    # Sheet 2 — one row per fund (confirmed hits first)
    if len(results_df) > 0:
        deduped = (results_df.sort_values("needs_review")
                             .drop_duplicates(subset=ID_COL, keep="first"))
        deduped.to_excel(writer, sheet_name="Funds_Deduped", index=False)

    # Sheet 3 — summary by concept
    if len(results_df) > 0:
        summary_rows = []
        for label, grp in results_df.groupby("matched_pattern_label"):
            summary_rows.append({
                "Concept":          grp["matched_concept"].iloc[0],
                "Pattern Label":    label,
                "Funds Matched":    grp[ID_COL].nunique(),
                "Needs Review (n)": int(grp["needs_review"].sum()),
                "Example Name":     grp[NAME_COL].iloc[0],
            })
        (pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)
            .to_excel(writer, sheet_name="Summary", index=False))

    # Sheet 4 — token scan (unknown acronyms to consider for Cell 3)
    abbrev_rows = [{"Token": t, "Count": c, "Known Expansion": ABBREV_EXPANSION.get(t, "UNKNOWN")}
                   for t, c in unknown_abbrevs[:100]]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

print(f"\nOutput written to:\n  {outfile}")
